In [1]:
import pandas as pd


def load_data(path):
    df = pd.read_csv(path, parse_dates=["timestamp"])
    return df


def feature_engineer(df):
    df = df.sort_values(["sensor_id", "timestamp"]).reset_index(drop=True)

    df["minute"] = df["timestamp"].dt.minute
    df["hour"] = df["timestamp"].dt.hour

    df["pressure_diff"] = (
        df.groupby("sensor_id")["pressure_psi"].diff().fillna(0)
    )
    df["flow_diff"] = (
        df.groupby("sensor_id")["flow_lpm"].diff().fillna(0)
    )

    # Rolling features (window = 5 readings per sensor)
    df["pressure_roll_mean"] = (
        df.groupby("sensor_id")["pressure_psi"]
        .rolling(window=5, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["flow_roll_mean"] = (
        df.groupby("sensor_id")["flow_lpm"]
        .rolling(window=5, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["pressure_roll_std"] = (
        df.groupby("sensor_id")["pressure_psi"]
        .rolling(window=5, min_periods=1)
        .std()
        .reset_index(level=0, drop=True)
        .fillna(0)
    )

    df["consumption_lpm"] = df["flow_lpm"]

    # Updated fillna usage (method parameter is deprecated)
    df = df.ffill().fillna(0)

    return df


def get_feature_matrix(df, feature_cols=None):
    if feature_cols is None:
        feature_cols = [
            "pressure_psi",
            "flow_lpm",
            "temperature_c",
            "pressure_roll_mean",
            "flow_roll_mean",
            "pressure_roll_std",
            "valve_state",
            "pressure_diff",
            "flow_diff",
            "hour",
            "minute"
        ]

    return df[feature_cols].values, feature_cols


if __name__ == "__main__":
    # Direct filename avoids issues in Jupyter/Colab where sys.argv may contain '-f'
    path = "sensor_readings.csv"

    df = load_data(path)
    df = feature_engineer(df)
    X, cols = get_feature_matrix(df)

    print("Loaded", df.shape, "features:", cols)


Loaded (20000, 16) features: ['pressure_psi', 'flow_lpm', 'temperature_c', 'pressure_roll_mean', 'flow_roll_mean', 'pressure_roll_std', 'valve_state', 'pressure_diff', 'flow_diff', 'hour', 'minute']


In [2]:
# model_autoencoder.py

import numpy as np
import pandas as pd
import joblib


# --- Functions from preprocessing ---

def load_data(path):
    df = pd.read_csv(path, parse_dates=["timestamp"])
    return df


def feature_engineer(df):
    df = df.sort_values(["sensor_id", "timestamp"]).reset_index(drop=True)

    df["minute"] = df["timestamp"].dt.minute
    df["hour"] = df["timestamp"].dt.hour

    df["pressure_diff"] = (
        df.groupby("sensor_id")["pressure_psi"].diff().fillna(0)
    )
    df["flow_diff"] = (
        df.groupby("sensor_id")["flow_lpm"].diff().fillna(0)
    )

    df["consumption_lpm"] = df["flow_lpm"]

    # Updated deprecated fillna usage
    df = df.ffill().fillna(0)

    return df


def get_feature_matrix(df, feature_cols=None):
    if feature_cols is None:
        feature_cols = [
            "pressure_psi",
            "flow_lpm",
            "temperature_c",
            "valve_state",
            "pressure_diff",
            "flow_diff",
            "hour",
            "minute"
        ]

    return df[feature_cols].values, feature_cols


# --- End of preprocessing functions ---


def build_autoencoder(input_dim):
    from tensorflow.keras import layers, Model, Input

    inputs = Input(shape=(input_dim,))
    x = layers.Dense(max(4, int(input_dim * 0.75)), activation="relu")(inputs)
    x = layers.Dense(max(4, int(input_dim * 0.5)), activation="relu")(x)

    bottleneck = layers.Dense(max(2, int(input_dim * 0.25)), activation="relu")(x)

    x = layers.Dense(max(4, int(input_dim * 0.5)), activation="relu")(bottleneck)
    x = layers.Dense(max(4, int(input_dim * 0.75)), activation="relu")(x)

    outputs = layers.Dense(input_dim, activation="linear")(x)

    autoencoder = Model(inputs, outputs, name="autoencoder")
    autoencoder.compile(optimizer="adam", loss="mse")

    return autoencoder


def train_autoencoder(
    csv_path,
    save_path="autoencoder.h5",
    epochs=10,
    batch_size=256
):
    from sklearn.preprocessing import StandardScaler
    import tensorflow as tf

    # Load & preprocess
    df = load_data(csv_path)
    df = feature_engineer(df)
    X, cols = get_feature_matrix(df)

    # Scale
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    # Build & train
    auto = build_autoencoder(Xs.shape[1])
    auto.fit(
        Xs,
        Xs,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.1,
        verbose=2
    )

    # Save artifacts
    auto.save(save_path)
    joblib.dump(scaler, save_path + ".scaler.pkl")
    joblib.dump(cols, save_path + ".cols.pkl")

    print("Saved autoencoder to", save_path)

    return auto, scaler, cols, df


if __name__ == "__main__":
    auto, scaler, cols, df = train_autoencoder(
        "sensor_readings.csv",
        epochs=5
    )


c:\Users\roat9\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Epoch 1/5
71/71 - 3s - 43ms/step - loss: 0.9928 - val_loss: 0.9230
Epoch 2/5
71/71 - 0s - 6ms/step - loss: 0.9497 - val_loss: 0.8702
Epoch 3/5
71/71 - 0s - 5ms/step - loss: 0.9048 - val_loss: 0.8430
Epoch 4/5
71/71 - 0s - 4ms/step - loss: 0.8887 - val_loss: 0.8365
Epoch 5/5
71/71 - 0s - 4ms/step - loss: 0.8780 - val_loss: 0.8258


Saved autoencoder to autoencoder.h5


In [3]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest


# ---------- Isolation Forest Prediction ----------
def predict_iforest(model, cols, df):
    X = df[cols].values
    scores = model.decision_function(X)
    preds = model.predict(X)

    df["if_score"] = scores
    df["if_label"] = np.where(preds == -1, 1, 0)  # 1 = anomaly

    return df



# ---------- Load or Train Isolation Forest ----------
def load_or_train_iforest(df, feature_cols, model_path="isolation_forest.pkl"):

    if os.path.exists(model_path):
        print("Loading Isolation Forest model...")
        model, cols = joblib.load(model_path)

    else:
        print("Isolation Forest model not found. Training new model...")

        X = df[feature_cols].values

        model = IsolationForest(
            n_estimators=200,
            contamination=0.05,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X)

        joblib.dump((model, feature_cols), model_path)
        cols = feature_cols

        print("Isolation Forest model saved as", model_path)

    return model, cols


# ---------- Feature Columns ----------
if_cols = [
    "pressure_psi",
    "flow_lpm",
    "temperature_c",
    "valve_state",
    "pressure_diff",
    "flow_diff",
    "hour",
    "minute"
]


# ---------- Run Isolation Forest ----------
model, if_cols = load_or_train_iforest(df, if_cols)
df = predict_iforest(model, if_cols, df)


# ---------- Results ----------
print("Potential leaks identified by Isolation Forest:")

leaks_df = df[df["if_label"] == 1]

if not leaks_df.empty:
    print(
        leaks_df[
            [
                "timestamp",
                "sensor_id",
                "pressure_psi",
                "flow_lpm",
                "if_score",
                "is_leak_sim"
            ]
        ]
    )
    print(f"\nTotal {len(leaks_df)} potential leaks detected.")
else:
    print("No leaks detected by Isolation Forest.")
    summary = (
    df.groupby("sensor_id")
      .agg(
          total_records=("sensor_id", "count"),
          leak_cases=("is_leak", "sum"),
          avg_severity=("leak_severity", "mean")
      )
      .reset_index()
)



Loading Isolation Forest model...
Potential leaks identified by Isolation Forest:
                       timestamp  sensor_id  pressure_psi  flow_lpm  if_score  \
9     2025-12-01 15:17:40.991388          1        40.980     9.139 -0.067703   
32    2025-12-01 17:12:40.991388          1        39.878     8.380 -0.009099   
76    2025-12-01 20:52:40.991388          1        40.334     7.357 -0.029644   
100   2025-12-01 22:52:40.991388          1        33.248    15.996 -0.188828   
101   2025-12-01 22:57:40.991388          1        33.400    17.653 -0.090920   
...                          ...        ...           ...       ...       ...   
19858 2025-12-15 00:02:40.991388          5        38.964     9.167 -0.009006   
19902 2025-12-15 03:42:40.991388          5        39.423     9.417 -0.054464   
19938 2025-12-15 06:42:40.991388          5        36.954     9.528 -0.004624   
19942 2025-12-15 07:02:40.991388          5        40.329     8.952 -0.010391   
19946 2025-12-15 07:22:40.9

In [ ]:
import gradio as gr
import pandas as pd
import joblib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

# =========================================================
# DATA LOADING
# =========================================================

def load_data(file_obj):
    return pd.read_csv(file_obj.name, parse_dates=["timestamp"])


# =========================================================
# FEATURE ENGINEERING
# =========================================================

def feature_engineer(df):
    df = df.sort_values(["sensor_id", "timestamp"]).reset_index(drop=True)

    df["hour"] = df["timestamp"].dt.hour
    df["minute"] = df["timestamp"].dt.minute

    df["pressure_diff"] = df.groupby("sensor_id")["pressure_psi"].diff().fillna(0)
    df["flow_diff"] = df.groupby("sensor_id")["flow_lpm"].diff().fillna(0)

    df = df.ffill().fillna(0)
    return df


# =========================================================
# PREDICTION
# =========================================================

def predict_iforest(model, cols, df):
    X = df[cols].values
    scores = model.decision_function(X)
    preds = model.predict(X)

    df["if_score"] = scores
    df["if_label"] = np.where(preds == -1, 1, 0)

    # 🔥 Leak severity (0–100)
    df["leak_severity"] = (
        (df["if_score"].max() - df["if_score"]) /
        (df["if_score"].max() - df["if_score"].min() + 1e-6)
    ) * 100

    return df


# =========================================================
# PLOT
# =========================================================

def plot_pressure(df):
    fig, ax = plt.subplots(figsize=(11, 4.5))

    # Separate normal and leak points
    normal = df[df["if_label"] == 0]
    leaks = df[df["if_label"] == 1]

    # Normal pressure trend
    ax.plot(
        normal["timestamp"],
        normal["pressure_psi"],
        label="Normal Operation",
        linewidth=2.4,
        alpha=0.85
    )

    # Leak points
    ax.scatter(
        leaks["timestamp"],
        leaks["pressure_psi"],
        label="Detected Leak",
        color="red",
        s=50,
        edgecolor="black",
        linewidth=0.6,
        zorder=3
    )

    # Titles and labels
    ax.set_title(
        "Pressure Time Series with Leak Anomaly Detection",
        fontsize=14,
        fontweight="bold",
        pad=12
    )
    ax.set_xlabel("Timestamp", fontsize=11)
    ax.set_ylabel("Pressure (psi)", fontsize=11)

    # Grid styling
    ax.grid(
        which="major",
        linestyle="--",
        linewidth=0.6,
        alpha=0.4
    )

    # Legend styling
    ax.legend(
        loc="upper right",
        frameon=True,
        framealpha=0.95,
        fontsize=10
    )

    # Clean spacing
    fig.tight_layout()

    return fig


# =========================================================
# LOAD MODEL
# =========================================================

try:
    if_model, if_cols = joblib.load("isolation_forest.pkl")
    model_status = "✅ Model Loaded Successfully"
except:
    if_model = None
    if_cols = None
    model_status = "❌ Model Not Found"


# =========================================================
# MAIN FUNCTION
# =========================================================

def run_app(file):

    if if_model is None or file is None:
        return None, None, None, None, None

    df = load_data(file)
    df = feature_engineer(df)
    df = predict_iforest(if_model, if_cols, df)

    fig = plot_pressure(df)

    leak_df = df[df["if_label"] == 1][
        ["timestamp", "sensor_id", "pressure_psi", "flow_lpm", "leak_severity"]
    ]

    # 📊 Metrics
    total = len(df)
    leaks = len(leak_df)
    avg_severity = round(leak_df["leak_severity"].mean(), 2) if leaks > 0 else 0

    # 📍 Sensor-wise summary
    sensor_summary = (
        leak_df.groupby("sensor_id")
        .size()
        .reset_index(name="Leak Count")
    )

    return (
        fig,
        leak_df,
        f"📊 Total Records: {total}",
        f"🚨 Leaks Detected: {leaks}",
        f"🔥 Avg Severity: {avg_severity}",
        sensor_summary
    )


# =========================================================
# BEAUTIFUL UI
# =========================================================

with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown(
        """
        # 💧 Smart Water Leakage Detection Dashboard
        ### Isolation Forest–Based Anomaly Detection System
        _Industrial IoT · Smart Cities · Predictive Maintenance_
        """
    )

    gr.Markdown(f"**Model Status:** {model_status}")

    with gr.Tabs():

        # ================= DASHBOARD TAB =================
        with gr.Tab("📊 Dashboard"):

            with gr.Row():
                file_input = gr.File(label="📂 Upload Sensor CSV", file_types=[".csv"])
                run_btn = gr.Button("🚀 Run Detection", variant="primary")

            with gr.Row():
                metric1 = gr.Markdown()
                metric2 = gr.Markdown()
                metric3 = gr.Markdown()

            plot_out = gr.Plot(label="Pressure Analysis")

            table_out = gr.Dataframe(label="Detected Leaks", wrap=True)
            sensor_table = gr.Dataframe(label="Sensor-wise Leak Count")

        # ================= DATA PREVIEW =================
        with gr.Tab("📄 Data Preview"):
            preview = gr.Dataframe(label="Uploaded Data Preview")

        # ================= ABOUT =================
        with gr.Tab("ℹ️ About"):
            gr.Markdown(
                """
                ### 🔍 About This Project
                - **Algorithm:** Isolation Forest  
                - **Purpose:** Detect water leaks using anomaly detection  
                - **Output:** Leak points, severity score, visualization  

                **Applications:**
                - Smart Water Distribution  
                - Industrial Pipeline Monitoring  
                - Smart Cities Infrastructure
                """
            )

    run_btn.click(
        fn=run_app,
        inputs=file_input,
        outputs=[
            plot_out,
            table_out,
            metric1,
            metric2,
            metric3,
            sensor_table
        ]
    )

    file_input.change(
        lambda f: pd.read_csv(f.name).head(20) if f else None,
        inputs=file_input,
        outputs=preview
    )

    gr.Markdown(
        """
        ---
        **🛠 Tech Stack:** Python · Scikit-Learn · Pandas · Matplotlib · Gradio  *
        """
    )

app.launch(share=True, debug=True)


C:\Users\roat9\AppData\Local\Temp\ipykernel_17764\1800388706.py:176: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
